In [ ]:
# --- cell 1 · bootstrap（任务书 §0.1 末、§0.2）—— 五个 B notebook 里这一格完全相同 ---
import pathlib, sys

# (1) 向上搜索含 npnn/ 的目录 —— 不依赖 Jupyter 的启动目录
def _find_taskb_root(start):
    for p in (start, *start.parents):
        if (p / 'npnn').is_dir():
            return p
        if (p / 'task_b' / 'npnn').is_dir():      # 从项目根启动 Jupyter 的情况
            return p / 'task_b'
    return None

_here = pathlib.Path.cwd().resolve()
TASKB_ROOT = _find_taskb_root(_here)
assert TASKB_ROOT is not None, f'从 {_here} 逐级往上找不到含 npnn/ 的目录；请从 task_b/ 或项目根启动'
sys.path.insert(0, str(TASKB_ROOT))

# (2) 物理挡住「误 import A 的 src」：把项目根从 sys.path 里摘掉（任务书 §0.1.2）
_root = TASKB_ROOT.parent
sys.path[:] = [e for e in sys.path if pathlib.Path(e or '.').resolve() != _root]

# (3) 线程预算：必须在 import numpy 之前（BLAS 首次 import 时锁死线程数）
import config
config.limit_threads()

# (4) 环境留痕 —— 报告 B.1 要用：哪个解释器、哪个 numpy
import numpy as np
print('sys.executable:', sys.executable)
print('sys.prefix    :', sys.prefix)
print('numpy         :', np.__version__)
print('TASKB_ROOT    :', TASKB_ROOT)


# 13 · 最终分析与图表（第 4 阶段素材）

对应任务书：§4.2（图表清单）、§4.4（Run All 要求）。

**只读 `runs/` 里落盘的数据画图，绝不重新训练** —— 这样本 notebook 从头 Execute All 只要几秒。
（§4.4 专门警告过：B 的 notebook 互相依赖且训练很慢，所以图表一律从落盘文件重画。）

✍️ 产出（对着 §4.2 的 9 项清单打勾）：
- [ ] 数值梯度的 h 扫描曲线（对数轴）
- [ ] 模块验证汇总表（每模块 × 前向误差 / 反向误差 / 数值梯度误差）
- [ ] Conv2d 多形状对拍表（覆盖 stride / kernel / 不整除等边界）
- [ ] im2col vs 朴素循环耗时柱状图（+ numpy vs torch 速度比）
- [ ] 整网 loss 对拍图（L2 核心图：两条线 + 差值子图）
- [ ] numpy vs torch 精度曲线（两条 test acc）
- [ ] ★ SN 权重演化曲线（w_BN / w_IN / w_LN 随训练变化）
- [ ] SN 退化测试表（三行，各报 max|Δ|）
- [ ] 环境与版本留痕 + 纯度守卫测试的输出

图输出到 `report/figs/`（供 main.tex 引用）；原始数据留在 `runs/`。

⚠️ 所有误差都给量级（`max|Δ| = 3.2e-11` 这种），不要写「基本一致」。
「报告里的每个数字都能追溯到某次 run 或某个测试脚本」是提交清单的一条。


In [ ]:
# --- cell 2 · §3.5 的三问三答：精度对比 + 速度对比（只读落盘数据）---
import csv
import json
import matplotlib.pyplot as plt
from config import FIG_DIR, RUNS_DIR

def load_run(name):
    with open(RUNS_DIR / name / 'metrics.csv', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    return {k: [float(r[k]) for r in rows] for k in rows[0]}

nm = load_run('cnn_sn')          # numpy 实现
tm = load_run('torch_ref')       # torch 参照（同一份 tests/arch.py 的 ARCH）
ep = nm['epoch']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
ax1.plot(ep, nm['val_acc'], marker='o', ms=3, label='numpy')
ax1.plot(ep, tm['val_acc'], marker='s', ms=3, label='torch')
ax1.set_xlabel('epoch'); ax1.set_ylabel('val accuracy')
ax1.set_title('validation accuracy'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, nm['train_acc'], label='numpy train')
ax2.plot(ep, nm['val_acc'], label='numpy val')
ax2.plot(ep, tm['train_acc'], '--', label='torch train')
ax2.plot(ep, tm['val_acc'], '--', label='torch val')
ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy')
ax2.set_title('train vs val'); ax2.legend(fontsize=7); ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

cfg_n = json.loads((RUNS_DIR / 'cnn_sn' / 'config.json').read_text(encoding='utf-8'))
cfg_t = json.loads((RUNS_DIR / 'torch_ref' / 'config.json').read_text(encoding='utf-8'))
l2 = json.loads((RUNS_DIR / 'l2' / 'parity.json').read_text(encoding='utf-8'))

mean_n = np.mean(nm['epoch_time'])
mean_t = np.mean(tm['epoch_time'])
print(f'{"":26s}{"numpy":>12s}{"torch":>12s}')
print(f'{"params":26s}{cfg_n["n_params"]:>12,}{cfg_n["n_params"]:>12,}   (形状逐一相同，见 tests/arch.py)')
print(f'{"epochs":26s}{cfg_n["epochs"]:>12d}{cfg_t["epochs"]:>12d}')
print(f'{"final train acc":26s}{nm["train_acc"][-1]:>11.1%}{tm["train_acc"][-1]:>11.1%}')
print(f'{"final val acc":26s}{nm["val_acc"][-1]:>11.1%}{tm["val_acc"][-1]:>11.1%}')
print(f'{"mean epoch time":26s}{mean_n:>10.1f}s{mean_t:>11.1f}s')
print(f'{"total train time":26s}{cfg_n["train_time_s"]/60:>9.1f}m{cfg_t["train_time_s"]/60:>10.1f}m')
print(f'{"speed ratio (np/torch)":26s}{mean_n/mean_t:>22.1f}x')
print()
print('Q1 精度是否相近:  |Δval acc| = '
      f'{abs(nm["val_acc"][-1] - tm["val_acc"][-1]):.2%}')
print(f'Q2 轨迹是否一致:  L2 整网对拍 max|Δloss| = {l2["max_loss_diff"]:.2e}（判据 <1e-8）')
print(f'Q3 速度差多少:    numpy 每个 epoch 慢 {mean_n/mean_t:.1f} 倍')